<a href="https://colab.research.google.com/github/Shumookh/KSA/blob/main/ksa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import cv2, numpy as np, math
from pathlib import Path

LOGO_PATH = "ksa1.jpg"
WIDTH, HEIGHT, FPS = 1080, 1350, 60
N_PARTICLES = 60000
OUTPUT = "ksa.mp4"

BG = np.array([2, 10, 7], np.uint8)
GREEN = np.array([28, 135, 65], np.uint8)       # BGR dark green
GREEN_BRIGHT = np.array([45, 185, 90], np.uint8)
GOLD = np.array([55, 185, 225], np.uint8)       # BGR warm gold
GOLD_BRIGHT = np.array([80, 225, 255], np.uint8)
GREEN_GLOW = np.array([15, 85, 40], np.uint8)
GOLD_GLOW = np.array([45, 120, 160], np.uint8)

T0, T1, T2, T3, T4, T5, T6 = 1.4, 3.2, .8, 1.6, 3.2, 2.2, 2.6
TOTAL = T0 + T1 + T2 + T3 + T4 + T5 + T6

def ease(t):
    t = np.clip(t, 0, 1)
    return t*t*(3-2*t)

def load_logo():
    p = Path(LOGO_PATH)
    if not p.exists(): raise FileNotFoundError(f"Logo not found: {p.resolve()}")
    im = cv2.imread(str(p))
    if im is None: raise RuntimeError("Cannot read ksa.jpeg")
    h,w = im.shape[:2]
    s = min(WIDTH*.78/w, HEIGHT*.62/h)
    nw,nh = int(w*s),int(h*s)
    im = cv2.resize(im,(nw,nh),interpolation=cv2.INTER_LANCZOS4)
    gray = cv2.cvtColor(im,cv2.COLOR_BGR2GRAY)
    mask = (gray < 235).astype(np.uint8)
    mask = cv2.morphologyEx(mask,cv2.MORPH_OPEN,np.ones((2,2),np.uint8))
    mask = cv2.morphologyEx(mask,cv2.MORPH_CLOSE,np.ones((3,3),np.uint8))
    mask = cv2.dilate(mask,np.ones((2,2),np.uint8),1)
    y,x = np.where(mask>0)
    if len(x)==0: raise RuntimeError("No logo detected.")
    ox,oy=(WIDTH-nw)/2,(HEIGHT-nh)/2
    base=np.column_stack([x.astype(np.float32)+ox,y.astype(np.float32)+oy])
    ids=np.random.choice(len(base),N_PARTICLES,replace=True)
    return base[ids].astype(np.float32)

class ParticleSystem:
    def __init__(self,target):
        self.target=target
        self.n=len(target)
        self.c=np.array([WIDTH/2,HEIGHT/2],np.float32)
        self.a0=np.random.uniform(0,2*np.pi,self.n)
        self.r0=np.random.uniform(180,780,self.n)
        self.start=np.column_stack([self.c[0]+np.cos(self.a0)*self.r0,
                                    self.c[1]+np.sin(self.a0)*self.r0]).astype(np.float32)
        q=target-self.c
        self.ta=np.arctan2(q[:,1],q[:,0])
        self.tr=np.linalg.norm(q,axis=1)
        self.turns=np.random.uniform(1.7,4.0,self.n)
        self.phase=np.random.uniform(0,2*np.pi,self.n)
        self.rand=np.column_stack([np.cos(self.phase),np.sin(self.phase)]).astype(np.float32)

    def pos(self,t):
        s0=T0; s1=s0+T1; s2=s1+T2; s3=s2+T3; s4=s3+T4; s5=s4+T5

        if t<s0:
            return self.start+np.column_stack([
                np.sin(self.phase+t*3)*28,
                np.cos(self.phase*1.3+t*2.2)*28])

        if t<s1:
            p=ease((t-s0)/T1)
            # Strong inward spiral
            ang=self.a0+self.turns*2*np.pi*p
            rad=self.r0*(1-p)+self.tr*p
            spiral=np.column_stack([self.c[0]+np.cos(ang)*rad,
                                    self.c[1]+np.sin(ang)*rad])
            return spiral*(1-p)**0.65 + self.target*(1-(1-p)**0.65)

        if t<s2:
            p=ease((t-s1)/T2)
            jitter=np.column_stack([np.sin(self.phase+t*20),
                                    np.cos(self.phase*1.7+t*18)])
            return self.target+jitter*2.2*(1-p)

        if t<s3:
            p=(t-s2)/T3
            scale=1+.006*np.sin(p*np.pi*4)
            return (self.target-self.c)*scale+self.c

        if t<s4:
            p=(t-s3)/T4
            q=self.target-self.c
            ang=p*2*np.pi
            z=np.sin(self.target[:,0]*.018)*85
            x=q[:,0]*np.cos(ang)+z*np.sin(ang)
            y=q[:,1]
            # Spiral orbit while rotating
            extra=np.sin(p*np.pi)*20
            x+=np.cos(self.ta+p*2*np.pi)*extra
            y+=np.sin(self.ta+p*2*np.pi)*extra
            perspective=1/(1+z*.0008)
            return np.column_stack([x*perspective,y*perspective])+self.c

        if t<s5:
            p=ease((t-s4)/T5)
            ang=self.ta+p*6*np.pi
            rad=self.tr+p*900
            spiral=np.column_stack([self.c[0]+np.cos(ang)*rad,
                                    self.c[1]+np.sin(ang)*rad])
            return self.target*(1-p)+spiral*p

        p=ease((t-s5)/T6)
        ang=self.ta+5*np.pi
        start=np.column_stack([self.c[0]+np.cos(ang)*(self.tr+700),
                               self.c[1]+np.sin(ang)*(self.tr+700)])
        return start*(1-p)+self.target*p

def draw(frame,pts,t,intense):
    valid=(pts[:,0]>=0)&(pts[:,0]<WIDTH)&(pts[:,1]>=0)&(pts[:,1]<HEIGHT)
    p=pts[valid].astype(np.int32)
    if len(p)==0:return
    idx=np.arange(len(p))
    gold=((idx*17+int(t*35))%100)<28
    gp=p[gold]; gr=p[~gold]

    gg=np.zeros_like(frame); ag=np.zeros_like(frame)
    if len(gr): gg[gr[:,1],gr[:,0]]=GREEN_GLOW
    if len(gp): ag[gp[:,1],gp[:,0]]=GOLD_GLOW
    gg=cv2.GaussianBlur(gg,(0,0),7); ag=cv2.GaussianBlur(ag,(0,0),6)
    frame[:]=cv2.addWeighted(frame,1,gg,.55 if intense else .30,0)
    frame[:]=cv2.addWeighted(frame,1,ag,.75 if intense else .40,0)

    core=np.zeros_like(frame)
    if len(gr): core[gr[:,1],gr[:,0]]=GREEN_BRIGHT if intense else GREEN
    if len(gp): core[gp[:,1],gp[:,0]]=GOLD_BRIGHT if intense else GOLD
    if intense:
        core=cv2.dilate(core,np.ones((2,2),np.uint8),1)
        core=cv2.morphologyEx(core,cv2.MORPH_CLOSE,np.ones((2,2),np.uint8))
    frame[:]=cv2.addWeighted(frame,1,core,.98,0)

def main():
    np.random.seed(42)
    print(f"Particles: {N_PARTICLES:,} | {WIDTH}x{HEIGHT} | {FPS} FPS")
    target=load_logo()
    ps=ParticleSystem(target)
    writer=cv2.VideoWriter(OUTPUT,cv2.VideoWriter_fourcc(*"mp4v"),FPS,(WIDTH,HEIGHT))
    if not writer.isOpened(): raise RuntimeError("Could not create video.")
    start=T0+T1; end=start+T2+T3
    for i in range(int(TOTAL*FPS)):
        t=i/FPS
        frame=np.full((HEIGHT,WIDTH,3),BG,np.uint8)
        draw(frame,ps.pos(t),t,start<=t<=end)
        writer.write(frame)
        if i%FPS==0: print(f"Rendering {t:.1f}/{TOTAL:.1f}s")
    writer.release()
    print(f"DONE: {Path(OUTPUT).resolve()}")

if __name__=="__main__":
    main()

Particles: 60,000 | 1080x1350 | 60 FPS
Rendering 0.0/15.0s
Rendering 1.0/15.0s
Rendering 2.0/15.0s
Rendering 3.0/15.0s
Rendering 4.0/15.0s
Rendering 5.0/15.0s
Rendering 6.0/15.0s
Rendering 7.0/15.0s
Rendering 8.0/15.0s
Rendering 9.0/15.0s
Rendering 10.0/15.0s
Rendering 11.0/15.0s
Rendering 12.0/15.0s
Rendering 13.0/15.0s
Rendering 14.0/15.0s
DONE: /content/ksa.mp4
